In [ ]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

import os
os.environ["HF_HOME"] = "/shared/data3/pk36/.cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"

import argparse
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from dataclasses import dataclass
import json_repair
import re
import json
from typing import List, Dict
from tqdm import tqdm
from collections import defaultdict
import time

from search import search_semantic_scholar, collect_snippets
from prompts import (
    create_initial_decomposition_prompt,
    create_target_domain_analysis_prompt,
    create_cross_domain_query_prompt,
    create_cross_domain_analysis_prompt,
    initial_decomposition_schema,
    target_domain_analysis_schema,
    cross_domain_queries_schema,
    cross_domain_analysis_schema
)
from classes import ResearchProblem, Question, Domain
from utils import prepare_output
from main import batch_llm_inference, retrieve_papers_for_question

/home/pk36/structured_survey/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-22 21:26:58 [__init__.py:216] Automatically detected platform cuda.


## Setup

In [2]:
@dataclass
class Args:
    problem_file: str = "data/claimspect.txt"
    target_domain: str = "Computer Science"
    model_name: str = "Qwen/Qwen3-14B"
    output_dir: str = "output_debug"
    max_papers_per_query: int = 20

args = Args()

In [3]:
# Read problem statement
if os.path.exists(args.problem_file):
    with open(args.problem_file, "r") as f:
        problem_file_text = f.read()
        match = re.search(r"Problem Statement:\s*(.*)", problem_file_text)
        if match:
            problem_statement = match.group(1).strip()
        else:
            print("Could not find problem statement in file!")
    print(f"Problem Statement: {problem_statement}\n")
else:
    print(f"File {args.problem_file} does not exist!")

# Create output file path
output_file_name = os.path.splitext(os.path.basename(args.problem_file))[0] + f"_{args.max_papers_per_query}_results.json"
condensed_output_file_name = os.path.splitext(os.path.basename(args.problem_file))[0] + f"_{args.max_papers_per_query}_condensed.json"

args.output_file = os.path.join(args.output_dir, output_file_name)
args.condensed_output_file = os.path.join(args.output_dir, condensed_output_file_name)

# Create output directory if needed
os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

Problem Statement: Public-facing scientific and political discourse is increasingly framed through concise claims that mask underlying complexity. Such claims—common in biomedical safety, public policy, and geopolitics—are rarely cleanly “true” or “false.” Instead, they depend on multiple interacting aspects (e.g., efficacy, safety, long-term risk, logistics), each supported by uneven and sometimes conflicting evidence. Existing fact-checking and stance-detection systems largely treat claims as monolithic units, assigning a single veracity label or stance per document. This document-level framing fails to capture aspect-specific disagreement, partial consensus, or gaps in existing research.



In [4]:
# Initialize vLLM model
print("Loading model...")
llm = LLM(model=args.model_name, tensor_parallel_size=2)
print("Model loaded.\n")

Loading model...
INFO 01-22 21:27:11 [utils.py:233] non-default args: {'tensor_parallel_size': 2, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-14B'}
INFO 01-22 21:27:12 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 01-22 21:27:12 [model.py:1510] Using max model len 40960


2026-01-22 21:27:12,610	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-22 21:27:12 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=3228006) INFO 01-22 21:27:13 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=3228006) INFO 01-22 21:27:13 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-14B', speculative_config=None, tokenizer='Qwen/Qwen3-14B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=ObservabilityConfig(show_hid

...

(EngineCore_DP0 pid=3228006) (Worker_TP0 pid=3228028) INFO 01-22 21:27:23 [default_loader.py:267] Loading weights took 4.20 seconds
(EngineCore_DP0 pid=3228006) (Worker_TP1 pid=3228032) INFO 01-22 21:27:23 [default_loader.py:267] Loading weights took 4.36 seconds
(EngineCore_DP0 pid=3228006) (Worker_TP0 pid=3228028) INFO 01-22 21:27:23 [gpu_model_runner.py:2653] Model loading took 13.8818 GiB and 4.821419 seconds
(EngineCore_DP0 pid=3228006) (Worker_TP1 pid=3228032) INFO 01-22 21:27:24 [gpu_model_runner.py:2653] Model loading took 13.8818 GiB and 5.141082 seconds
(EngineCore_DP0 pid=3228006) (EngineCore_DP0 pid=3228006) (Worker_TP1 pid=3228032) (Worker_TP0 pid=3228028) INFO 01-22 21:27:32 [backends.py:548] Using cache directory: /home/pk36/.cache/vllm/torch_compile_cache/69891fa5b4/rank_0_0/backbone for vLLM's torch.compile
INFO 01-22 21:27:32 [backends.py:548] Using cache directory: /home/pk36/.cache/vllm/torch_compile_cache/69891fa5b4/rank_1_0/backbone for vLLM's torch.compile
(Engin

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:05<00:00, 12.77it/s]
Capturing CUDA graphs (decode, FULL):  91%|█████████▏| 32/35 [00:02<00:00, 13.90it/s]

(EngineCore_DP0 pid=3228006) (Worker_TP1 pid=3228032) INFO 01-22 21:27:48 [custom_all_reduce.py:203] Registering 8262 cuda graph addresses


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 13.79it/s]


(EngineCore_DP0 pid=3228006) (Worker_TP0 pid=3228028) INFO 01-22 21:27:48 [custom_all_reduce.py:203] Registering 8262 cuda graph addresses
(EngineCore_DP0 pid=3228006) (Worker_TP1 pid=3228032) INFO 01-22 21:27:49 [gpu_model_runner.py:3480] Graph capturing finished in 9 secs, took 0.97 GiB
(EngineCore_DP0 pid=3228006) (Worker_TP0 pid=3228028) INFO 01-22 21:27:49 [gpu_model_runner.py:3480] Graph capturing finished in 9 secs, took 0.97 GiB
(EngineCore_DP0 pid=3228006) INFO 01-22 21:27:49 [core.py:210] init engine (profile, create kv cache, warmup model) took 24.94 seconds
INFO 01-22 21:27:50 [llm.py:306] Supported_tasks: ['generate']
Model loaded.



## Decomposition

In [5]:
prompt = create_initial_decomposition_prompt(problem_statement, args.target_domain)
messages = [{"role": "user", "content": prompt}]

decomposition_outputs = batch_llm_inference(
    llm, 
    [messages], 
    initial_decomposition_schema,
    temperature=0.7
)
decomposition_output = decomposition_outputs[0]

if decomposition_output is None:
    print("Failed to get decomposition output!")
else:
    print(json.dumps(decomposition_output, indent=2))

INFO 01-22 21:27:51 [chat_utils.py:560] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 1/1 [00:27<00:00, 27.81s/it, est. speed input: 43.37 toks/s, output: 40.35 toks/s]

{
  "problem_statement": "Public-facing scientific and political discourse is increasingly framed through concise claims that mask underlying complexity. Such claims\u2014common in biomedical safety, public policy, and geopolitics\u2014are rarely cleanly \u201ctrue\u201d or \u201cfalse.\u201d Instead, they depend on multiple interacting aspects (e.g., efficacy, safety, long-term risk, logistics), each supported by uneven and sometimes conflicting evidence. Existing fact-checking and stance-detection systems largely treat claims as monolithic units, assigning a single veracity label or stance per document. This document-level framing fails to capture aspect-specific disagreement, partial consensus, or gaps in existing research.",
  "target_domain": "Computer Science",
  "fine_grained_domain": "Natural Language Processing (NLP) - Claim Analysis and Multi-Aspect Sentiment/Veracity Detection",
  "core_challenge": "Current NLP systems for claim verification and stance detection operate at t

In [6]:
# Create ResearchProblem object
research_problem = ResearchProblem.from_initial_decomposition(
    decomposition_output, 
    args.target_domain
)

print(f"Generated {len(research_problem.research_questions)} research questions:")
for q in research_problem.research_questions:
    print(f"  - {q.id}:\n\t\t-{q.domain_specific_question}\n\t\t-{q.domain_agnostic_question}")
print()

Generated 5 research questions:
  - q1:
		-How can NLP models be trained to identify and disentangle multi-aspect claims, where each aspect may have its own veracity status and supporting evidence?
		-How can systems be designed to recognize and evaluate multiple facets of a single statement, each with potentially different truth values and supporting data?
  - q2:
		-What are the limitations of existing token- and sentence-level attention mechanisms in capturing the structured dependencies between claim aspects and their supporting evidence?
		-What are the constraints of current attention-based models in representing the relationships between different facets of a statement and their supporting information?
  - q3:
		-How can we incorporate structured knowledge from external sources (e.g., ontologies, evidence databases) to improve the alignment of claim aspects with their supporting evidence?
		-How can external structured knowledge be integrated to better match different parts of a

## Target Domain Analysis

In [7]:
# Step 2a: Retrieve papers for all questions in target domain
print("\n2a. Retrieving papers from target domain...")
for question in research_problem.research_questions:
    print(f"  Retrieving for {question.id}...")
    papers = retrieve_papers_for_question(
        question, 
        research_problem.target_domain,
        max_papers=args.max_papers_per_query
    )
    research_problem.target_domain.add_question_papers(question, papers)
    print(f"    -Retrieved {len(papers)} papers")


2a. Retrieving papers from target domain...
  Retrieving for q1...
	 -Searching Semantic Scholar for query: multi-aspect claim detection in domain: Computer Science.
	 -Searching Semantic Scholar for query: aspect disentanglement in NLP in domain: Computer Science.
	 -Searching Semantic Scholar for query: claim verification with fine-grained aspects in domain: Computer Science.
	 -Searching Semantic Scholar for query: document-level vs aspect-level verification in domain: Computer Science.
	 -Searching Semantic Scholar for query: evidence alignment for multi-aspect claims in domain: Computer Science.
    -Retrieved 17 papers
  Retrieving for q2...
	 -Searching Semantic Scholar for query: attention mechanisms for claim aspects in domain: Computer Science.
	 -Searching Semantic Scholar for query: structured attention in NLP in domain: Computer Science.
	 -Searching Semantic Scholar for query: limitations of attention for multi-aspect reasoning in domain: Computer Science.
	 -Searching S

In [8]:
# Step 2b: Batch analyze all questions in target domain
print("\n2b. Analyzing target domain papers (batch inference)...")

# Prepare batch of analysis prompts
analysis_messages_list = []
for question in research_problem.research_questions:
    papers = research_problem.target_domain.fetch_question_papers(question)
    
    if not papers:
        print(f"  Warning: No papers for {question.id}, skipping analysis")
        continue
    
    prompt = create_target_domain_analysis_prompt(
        research_problem=research_problem.problem_statement,
        domain_specific_question=question.domain_specific_question,
        domain_agnostic_question=question.domain_agnostic_question,
        question_rationale=question.rationale,
        papers_with_snippets=papers,
        target_domain=args.target_domain,
        fine_grained_domain=research_problem.fine_grained_domain
    )
    messages = [{"role": "user", "content": prompt}]
    analysis_messages_list.append(messages)

# Batch inference for all analyses
if analysis_messages_list:
    analysis_outputs = batch_llm_inference(
        llm,
        analysis_messages_list,
        target_domain_analysis_schema,
        temperature=0.5,  # Lower temperature for analysis,
        max_tokens=4096
    )


2b. Analyzing target domain papers (batch inference)...


Processed prompts: 100%|██████████| 5/5 [01:17<00:00, 15.40s/it, est. speed input: 523.90 toks/s, output: 140.61 toks/s]


In [9]:
# Process analysis results
for i, (question, analysis_output) in enumerate(zip(research_problem.research_questions, analysis_outputs)):
    if analysis_output is None:
        print(f"  Failed to analyze {question.id}")
        continue
    
    question.target_domain_analysis = analysis_output
    research_problem.target_domain.add_question_analysis(question, analysis_output)
    
    # Determine if addressed
    assessment = analysis_output.get("overall_assessment", "largely unaddressed").lower()
    is_addressed = "substantially" in assessment or "partial" in assessment
    question.mark_as_addressed(is_addressed)
    
    print(f"  {question.id}: {assessment} ({question.domain_specific_question})")
    
    # Create sub-questions for remaining challenges
    remaining_challenges = analysis_output.get("remaining_challenges", [])
    for challenge_data in remaining_challenges:
        challenge = research_problem.add_remaining_challenge(question, challenge_data)
        print(f"    -> New challenge: {challenge.domain_specific_question}")

  q1: partially addressed (How can NLP models be trained to identify and disentangle multi-aspect claims, where each aspect may have its own veracity status and supporting evidence?)
    -> New challenge: How can NLP models be trained to dynamically and contextually disentangle aspects of claims without predefined aspect categories?
    -> New challenge: How can NLP models be trained to evaluate the veracity of each aspect of a claim with varying levels of evidence and uncertainty?
  q2: partially addressed (What are the limitations of existing token- and sentence-level attention mechanisms in capturing the structured dependencies between claim aspects and their supporting evidence?)
    -> New challenge: How can attention mechanisms be designed to explicitly model the hierarchical and compositional structure of claims with multiple interacting aspects?
    -> New challenge: How can attention mechanisms be made more robust to conflicting or incomplete evidence when evaluating claim asp

In [10]:
for i, (question, analysis_output) in enumerate(zip(research_problem.research_questions, analysis_outputs)):
    print(f"  {question.id}: {assessment} ({question.domain_specific_question})")
    remaining_challenges = question.remaining_challenges
    for challenge in remaining_challenges:
        print(f"\t-> New challenge: {challenge.domain_specific_question}")
        print(f"\t\t-> Rationale: {challenge.rationale}")

  q1: largely unaddressed (How can NLP models be trained to identify and disentangle multi-aspect claims, where each aspect may have its own veracity status and supporting evidence?)
	-> New challenge: How can NLP models be trained to dynamically and contextually disentangle aspects of claims without predefined aspect categories?
		-> Rationale: This challenge is foundational because it limits the ability of NLP models to generalize and adapt to new types of claims and domains. Without this capability, the disentanglement of multi-aspect claims remains limited to predefined or manually curated aspects. Current approaches often rely on predefined aspects or require manual decomposition, which limits their ability to generalize across domains or handle novel claim structures. To solve this, models would need to dynamically identify and disentangle aspects based on the context of the claim, without prior assumptions about the structure of the claim.
	-> New challenge: How can NLP models b

## Cross-Domain Query Generation

In [11]:
# Get all questions needing cross-domain search
questions_needing_cross_domain = research_problem.get_questions_needing_cross_domain()

print(f"\nFound {len(questions_needing_cross_domain)} questions needing cross-domain search:")
for q in questions_needing_cross_domain:
    print(f"  - {q.id}: {q.domain_agnostic_question}")

if not questions_needing_cross_domain:
    print("\nAll questions addressed in target domain! No cross-domain search needed.")
else:
    # Step 3a: Generate cross-domain queries (batch)
    print("\n3a. Generating cross-domain queries (batch inference)...")
    
    cross_domain_messages_list = []
    for question in questions_needing_cross_domain:
        # Get target domain assessment if available
        target_assessment = None
        if question.parent_question and question.parent_question.target_domain_analysis:
            # This is a remaining challenge (includes )
            target_assessment = question.rationale
        elif question.target_domain_analysis:
            # This is an original question (iterate over all challenges in target domain analysis)
            target_assessment = ""
            for challenge in question.remaining_challenges:
                target_assessment += f"- {challenge.rationale}\n"
        
        prompt = create_cross_domain_query_prompt(
            problem_statement=research_problem.problem_statement,
            domain_specific_question=question.domain_specific_question,
            domain_agnostic_question=question.domain_agnostic_question,
            question_rationale=question.rationale,
            target_domain=args.target_domain,
            fine_grained_domain=research_problem.fine_grained_domain,
            target_domain_assessment=target_assessment
        )
        messages = [{"role": "user", "content": prompt}]
        cross_domain_messages_list.append(messages)
    
    # Batch inference for cross-domain queries
    cross_domain_outputs = batch_llm_inference(
        llm,
        cross_domain_messages_list,
        cross_domain_queries_schema,
        temperature=0.7
    )


Found 9 questions needing cross-domain search:
  - c1: How can systems be designed to dynamically identify and separate multiple facets of a statement without relying on predefined categories or assumptions about the structure of the statement?
  - c2: How can systems be designed to evaluate the truth value of different parts of a statement when the evidence supporting each part varies in quality and reliability?
  - c1: How can models be designed to represent the hierarchical and compositional relationships between different facets of a statement and their supporting information?
  - c2: How can models be designed to handle conflicting or incomplete information when representing the relationships between different facets of a statement and their supporting information?
  - c1: How can we iteratively refine the matching of different parts of a statement with their supporting evidence using structured data?
  - c2: How can we ensure that structured data used to match parts of a stateme

Processed prompts: 100%|██████████| 9/9 [00:10<00:00,  1.12s/it, est. speed input: 1030.64 toks/s, output: 261.38 toks/s]


In [12]:
# Process cross-domain query results
cross_domain_analysis_prompts = []
cross_domain_analysis_keys = []
for question, cross_domain_output in tqdm(zip(questions_needing_cross_domain, cross_domain_outputs), total=len(questions_needing_cross_domain)):
    if cross_domain_output is None:
        print(f"  Failed to generate cross-domain queries for {question.id}")
        continue
    
    question.cross_domain_queries = cross_domain_output
    
    print(f"\n  {question.domain_agnostic_question}:")
    for domain_search in cross_domain_output.get("cross_domain_searches", []):
        domain_name = domain_search["domain"]
        queries = domain_search["queries"]
        
        # Get or create domain
        domain = research_problem.get_or_create_domain(domain_name)
        domain.add_question_queries(question, queries)
        question.add_external_domain(domain)
        
        print(f"    - {domain_name}: {len(queries)} queries")
        papers = retrieve_papers_for_question(
            question,
            domain,
            max_papers=args.max_papers_per_query
        )
        
        # Conduct cross-domain analysis on domain papers
        cross_domain_analysis_prompt = create_cross_domain_analysis_prompt(
            problem_statement=research_problem.problem_statement,
            domain_specific_question=question.domain_specific_question,
            domain_agnostic_question=question.domain_agnostic_question,
            question_challenge=question.rationale,
            source_domain=domain_name,
            papers_with_snippets=papers,
            target_domain=research_problem.target_domain,
            fine_grained_domain=research_problem.fine_grained_domain
        )
        cross_domain_analysis_messages = [{"role": "user", "content": cross_domain_analysis_prompt}]
        cross_domain_analysis_prompts.append(cross_domain_analysis_messages)
        cross_domain_analysis_keys.append((question, domain))
        
        domain.add_question_papers(question, papers)
        domain_search["retrieved_papers"] = papers
        print(f"      -Retrieved {len(papers)} papers")


  How can systems be designed to dynamically identify and separate multiple facets of a statement without relying on predefined categories or assumptions about the structure of the statement?:
    - Psychology: 3 queries
	 -Searching Semantic Scholar for query: cognitive disentanglement in domain: Psychology.
	 -Searching Semantic Scholar for query: conceptual framing in domain: Psychology.
Rate limited. Retrying after 5 seconds...
	 -Searching Semantic Scholar for query: semantic decomposition in domain: Psychology.
      -Retrieved 12 papers
    - Political Science: 3 queries
	 -Searching Semantic Scholar for query: policy argument decomposition in domain: Political Science.
	 -Searching Semantic Scholar for query: ideological framing in domain: Political Science.
	 -Searching Semantic Scholar for query: multi-dimensional rhetoric in domain: Political Science.
      -Retrieved 13 papers
    - Linguistics: 3 queries
	 -Searching Semantic Scholar for query: semantic layering in domain

In [ ]:
# Reconstruct prompts for cross-domain analysis based on updated prompt template
reconstructed_cross_domain_analysis_prompts = []
for (question, domain) in cross_domain_analysis_keys:
    papers = domain.fetch_question_papers(question)
    
    cross_domain_analysis_prompt = create_cross_domain_analysis_prompt(
        problem_statement=research_problem.problem_statement,
        domain_specific_question=question.domain_specific_question,
        domain_agnostic_question=question.domain_agnostic_question,
        question_challenge=question.rationale,
        source_domain=domain.domain_name,
        papers_with_snippets=papers,
        target_domain=research_problem.target_domain,
        fine_grained_domain=research_problem.fine_grained_domain
    )
    cross_domain_analysis_messages = [{"role": "user", "content": cross_domain_analysis_prompt}]
    reconstructed_cross_domain_analysis_prompts.append(cross_domain_analysis_messages)

In [13]:
# Step 3b: Batch cross-domain analyses
print("\n3b. Analyzing cross-domain papers (batch inference)...")
if cross_domain_analysis_prompts:
    cross_domain_analysis_outputs = batch_llm_inference(
        llm,
        cross_domain_analysis_prompts,
        cross_domain_analysis_schema,
        temperature=0.5,
        max_tokens=4096
    )
    
    # # Process cross-domain analysis results
    # for (question, domain), analysis_output in zip(cross_domain_analysis_keys, cross_domain_analysis_outputs):
    #     if analysis_output is None:
    #         print(f"  Failed to analyze cross-domain papers for question '{question.id}' in domain '{domain.domain_name}'")
    #         continue
        
    #     question.add_cross_domain_analysis(domain, analysis_output)
    #     print(f"  Analyzed cross-domain papers for question '{question.id}' in domain '{domain.domain_name}'")


3b. Analyzing cross-domain papers (batch inference)...


Processed prompts: 100%|██████████| 27/27 [02:45<00:00,  6.14s/it, est. speed input: 1039.40 toks/s, output: 337.46 toks/s]


In [23]:
options = []
questions2domains = defaultdict(dict)
questions2domains["research_problem"] = research_problem.problem_statement
questions2domains["domain"] = research_problem.target_domain.domain_name
questions2domains["fine_grained_domain"] = research_problem.fine_grained_domain

for idx, ((q, domain), out) in enumerate(zip(cross_domain_analysis_keys, cross_domain_analysis_outputs)):
    if_relevant = [p["paper_title"] if p["directly_addresses_challenge"] else None for p in out["paper_relevance"]]
    num_relevant = sum([1 if p is not None else 0 for p in if_relevant])
    prop_relevant = num_relevant/len(if_relevant)
    options.append((idx, out["domain_specific_question"], out["source_domain"], f": {num_relevant}/{len(if_relevant)}", prop_relevant))

    # if out["domain_specific_question"] not in questions2domains:
    #     questions2domains[out["domain_specific_question"]] = []
    if (prop_relevant > 0.5) and (out["challenge_sufficiency_assessment"]["is_challenge_addressed"]):
        if out["domain_specific_question"] not in questions2domains:
            questions2domains[out["domain_specific_question"]]["parent_question"] = q.parent_question.domain_specific_question
            questions2domains[out["domain_specific_question"]]["rationale"] = q.rationale
        
        paper_info = {p.lower():snippets for p, snippets in domain.fetch_question_papers(q).items()}
        questions2domains[out["domain_specific_question"]][out["source_domain"]] = {'papers': {p:paper_info[p.lower()] for p in if_relevant if p is not None}, 'takeaways': out["solution_takeaways"], "remaining_challenge": out["challenge_sufficiency_assessment"]}

ranked_options = sorted(options, key=lambda x: x[-1], reverse=True)
for o in ranked_options:
    print(o)

(7, 'How can attention mechanisms be designed to explicitly model the hierarchical and compositional structure of claims with multiple interacting aspects?', 'Philosophy', ': 9/13', 0.6923076923076923)
(17, 'How can we ensure logical consistency and validity of structured knowledge integration in claim verification systems?', 'Mathematics', ': 9/15', 0.6)
(2, 'How can NLP models be trained to dynamically and contextually disentangle aspects of claims without predefined aspect categories?', 'Linguistics', ': 7/13', 0.5384615384615384)
(3, 'How can NLP models be trained to evaluate the veracity of each aspect of a claim with varying levels of evidence and uncertainty?', 'Law', ': 7/14', 0.5)
(24, 'How can we model the interaction between claim aspects, such as when one aspect’s truth value depends on another, in a way that is interpretable and aligned with human reasoning patterns?', 'Philosophy', ': 6/12', 0.5)
(19, 'How can we develop evaluation metrics that are both robust and general

In [24]:
with open(f"output_debug/{os.path.splitext(os.path.basename(args.problem_file))[0]}_recommendations.json", "w") as f:
    json.dump(questions2domains, fp=f, indent=2)